# ベイズ MMM（Colab 版）

上から順に、各セルの左の **▶ ボタン** を押してください。設定はセル右側のフォームで変更できます（コードは触らなくて OK）。

| ステップ | 内容 | 目安時間 |
|---|---|---|
| ① 準備 | ツールのダウンロードとインストール | 1〜2 分（初回のみ） |
| ② データ | サンプル or 自分の CSV をアップロード | すぐ |
| ③ 設定 | 列名・チャネル・計算速度など | すぐ |
| ④ 実行 | 推定してレポートを表示 | fast: 約 2〜5 分 / standard: 約 5〜15 分 |
| ⑤ シナリオ | 「TV を 1.5 倍にしたら？」を試算（再推定なし） | 数秒 |

> ⚠ Colab はブラウザを閉じる・一定時間操作しないとファイルが消えます。レポートと設定はダウンロードして保存してください。

In [ ]:
#@title ① 準備（最初に 1 回）
import os, sys, subprocess
REPO = "/content/claude-code-test"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "-b", "claude/laughing-albattani-3u9cqu",
                    "https://github.com/data-everyday/claude-code-test.git", REPO], check=True)
else:  # 以前ダウンロード済みなら最新版に更新
    subprocess.run(["git", "-C", REPO, "pull", "-q", "origin", "claude/laughing-albattani-3u9cqu"], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymc", "numpyro", "h5netcdf"], check=True)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print("✅ 準備完了")

In [ ]:
#@title ② データ
データ = "サンプルデータで試す" #@param ["サンプルデータで試す", "CSVをアップロード"]
import pandas as pd
os.makedirs("data", exist_ok=True)
if データ == "CSVをアップロード":
    from google.colab import files
    up = files.upload()
    if not up:
        raise SystemExit("ファイルが選択されませんでした")
    CSV_PATH = os.path.join("data", list(up)[0])
    with open(CSV_PATH, "wb") as f:
        f.write(list(up.values())[0])
else:
    from mmm.sample_data import generate
    CSV_PATH = "data/sample_mmm.csv"
    generate().to_csv(CSV_PATH, index=False)
df_preview = pd.read_csv(CSV_PATH)
print(f"✅ {CSV_PATH}（{len(df_preview)} 行）")
print("列名:", ", ".join(df_preview.columns))
df_preview.head()

In [ ]:
#@title ③ 設定（フォームを編集してから ▶）
#@markdown **列名**（② で表示された列名をそのまま入力）
日付の列 = "date" #@param {type:"string"}
#@markdown ⚠ 目的変数の列は、② で表示された列名のうち KPI（来場者数など）の列名に必ず書き換えてください
目的変数の列 = "sales" #@param {type:"string"}
#@markdown KPI の名前と単位（レポートの表示に使用。例：`来場者数` と `人`、`売上` と `円`）
KPIの名前 = "売上" #@param {type:"string"}
KPIの単位 = "円" #@param {type:"string"}
#@markdown **広告チャネル**：`名前:広告費の列名:adstock:曲線の形` をカンマ区切り
#@markdown - adstock：`geometric`（すぐ効く）/ `delayed`（遅れて効く。TV など）
#@markdown - 曲線の形（省略可）：`auto`（推定）/ `c`（C字に固定）/ `s`（S字に限定）
チャネル = "tv:tv_spend:delayed, search:search_spend:geometric, social:social_spend:geometric" #@param {type:"string"}
#@markdown **広告以外の要因**：`列名:符号`（符号は `free` / `positive` / `negative`）。なければ空欄
コントロール = "price:negative, promo:positive" #@param {type:"string"}
#@markdown ---
#@markdown **計算速度**：`fast`＝試行用（数分）/ `standard`＝通常 / `precise`＝最終報告用（時間がかかる）
計算速度 = "fast" #@param ["fast", "standard", "precise"]
#@markdown **ホールドアウト検証**：末尾 N 週を学習から外し、その期間の予測精度（MAPE・RMSE・R²）を確認（初期値 6 週。0 で省略。計算時間は約 2 倍）
ホールドアウト週数 = 6 #@param {type:"integer"}
#@markdown ---
#@markdown **モデル**
#@markdown トレンド：`linear`＝期間全体で 1 本の直線 / `changepoint`＝一定間隔で傾きが変われる折れ線 / `none`＝なし
トレンド = "linear" #@param ["linear", "changepoint", "none"]
#@markdown （changepoint のみ）変化点の間隔（週）と柔軟さ（大きいほど折れやすい。大きすぎると広告効果を吸収）
変化点の間隔週 = 26 #@param {type:"slider", min:8, max:52, step:1}
変化点の柔軟さ = 0.05 #@param {type:"number"}
#@markdown 季節性：繰り返しパターンの周期（データが 1 年未満なら「なし」か、四半期の繰り返しがある場合のみ「四半期」）
季節性 = "1年（52週）" #@param ["1年（52週）", "半年（26週）", "四半期（13週）", "なし"]
#@markdown ---
#@markdown **イベント期間のフラグ**（その週が期間に 1 日でも重なれば 1。季節性の波で捉えきれない短い山・谷を吸収）
年末年始 = True #@param {type:"boolean"}
GW = True #@param {type:"boolean"}
お盆 = True #@param {type:"boolean"}
#@markdown 任意の期間（セール等）：`名前=開始~終了, 開始~終了; 名前2=開始~終了`（例: `夏セール=2024-07-01~2024-07-14, 2025-07-01~2025-07-14`）
任意イベント = "" #@param {type:"string"}
#@markdown 最大ラグ：広告効果が最大何週先まで続き得るか（実際の長さはデータから推定）
最大ラグ週数 = 13 #@param {type:"slider", min:2, max:20, step:1}
#@markdown **広告貢献の事前想定**（データが弱いチャネルほど、この想定に近い結果になる）
#@markdown - `自動`：「KPI（売上・来場者数など）のうち広告で説明される割合」を想定として入れる。ROI 事前の中央値 = 想定シェア × KPI 合計 ÷ 広告費合計
#@markdown - `直接指定`：ROI（KPI 増分 ÷ 広告費）の中央値を入力
#@markdown
#@markdown 例：KPI が売上で、売上合計 10 億円・広告費合計 1 億円の場合（件数 KPI でも考え方は同じ）
#@markdown | 想定シェア | ROI 事前の中央値 | データが弱いときの広告貢献 |
#@markdown |---|---|---|
#@markdown | 0.1 | 1.0 | 約 10%（1 億円） |
#@markdown | 0.2（初期値） | 2.0 | 約 20%（2 億円） |
#@markdown | 0.3 | 3.0 | 約 30%（3 億円） |
#@markdown
#@markdown 幅：想定をどれだけ緩くするか。0.5＝強め（想定シェアの約 0.5〜2 倍）、1.0＝初期値（約 0.3〜5 倍）、1.5＝かなり緩い（約 0.25〜15 倍）。幅を大きくすると想定の中央もやや上がります。
#@markdown このセルを実行すると、入力した値での「事前の想定範囲」が下に表示されます。
ROI事前の決め方 = "自動" #@param ["自動", "直接指定"]
広告貢献シェアの想定 = 0.2 #@param {type:"number"}
ROI中央値_直接指定 = 1.0 #@param {type:"number"}
ROI事前_幅 = 1.0 #@param {type:"number"}
#@markdown ---
#@markdown **予算最適化**（総予算 0 = 実績どおり）
最適化の対象週数 = 52 #@param {type:"integer"}
総予算 = 0 #@param {type:"number"}
下限倍率 = 0.7 #@param {type:"number"}
上限倍率 = 1.3 #@param {type:"number"}
#@markdown ---
#@markdown **採算（任意）**：入力すると、利益から見た最適な週次出稿ライン（追加 1 円の粗利が 1 円になる点）をレポートに表示
#@markdown - KPI 1 単位の金額：平均販売単価など（KPI が件数なら 1 件あたりの売上、KPI が売上金額なら 1）。0 で使わない
#@markdown - 粗利率：0.3 = 30%。売上ベースで見るなら 1.0
KPI1単位の金額 = 0 #@param {type:"number"}
粗利率 = 1.0 #@param {type:"number"}
#@markdown ---
#@markdown **区間の幅**（レポートの「〇% 区間」。0.95 = 95%：真の値がこの範囲にある確率が 95%）
区間の幅 = 0.95 #@param [0.8, 0.9, 0.94, 0.95, 0.99] {type:"raw"}

from mmm.colab import build_config, save_yaml
cfg_dict = build_config(
    csv_path=CSV_PATH, date_column=日付の列, target_column=目的変数の列,
    kpi_name=KPIの名前, kpi_unit=KPIの単位,
    channels=チャネル, controls=コントロール, speed=計算速度, holdout_weeks=ホールドアウト週数,
    seasonality=季節性, trend=トレンド, max_lag=最大ラグ週数,
    events_builtin=[n for n, on in [("year_end", 年末年始), ("golden_week", GW), ("obon", お盆)] if on],
    events_custom=任意イベント,
    changepoint_interval=変化点の間隔週, changepoint_scale=変化点の柔軟さ,
    roi_prior_mode="auto" if ROI事前の決め方 == "自動" else "manual", ad_share=広告貢献シェアの想定,
    roi_prior_median=ROI中央値_直接指定, roi_prior_sd=ROI事前_幅,
    opt_period_weeks=最適化の対象週数, budget_lower=下限倍率, budget_upper=上限倍率,
    total_budget=総予算, interval=区間の幅, value_per_kpi=KPI1単位の金額, gross_margin=粗利率)
CONFIG = save_yaml(cfg_dict, "colab_config.yaml")
print("✅ 設定を保存しました:", CONFIG)
print("チャネル:", [c["name"] for c in cfg_dict["channels"]])
from mmm.colab import describe_prior
print()
print("【確認と事前の想定】")
print(describe_prior(CONFIG))

In [ ]:
#@title ④ 実行（推定 → レポート表示）
import subprocess, time
t0 = time.time()
proc = subprocess.Popen([sys.executable, "-m", "mmm", "run", "--config", CONFIG],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    # 進捗バーなどの細かい出力は省き、主要なログだけ表示
    if "Suppressed" in line:
        continue
    if any(k in line for k in ["サンプリング", "WARNING", "ERROR", "Error", "レポート", "ホールドアウト",
                               "split", "train", "test", "Traceback", "raise", "データ"]):
        print(line.rstrip())
proc.wait()
print(f"\n⏱ {time.time() - t0:.0f} 秒")
if proc.returncode != 0:
    raise SystemExit("❌ エラーで停止しました。上の赤字・ERROR 行を確認してください")
# Colab の表示スタイルの影響を受けないよう、レポートを独立した枠（iframe）に入れて表示する。
# 表が枠より広い場合は表の下に横スクロールバーが出ます。枠の中は縦にもスクロールできます。
import html as _html
from IPython.display import HTML, display
_report = open("mmm_output/report.html", encoding="utf-8").read()
display(HTML(f'<iframe srcdoc="{_html.escape(_report, quote=True)}" '
             'style="width:100%;height:1100px;border:1px solid #ddd;border-radius:6px"></iframe>'))
print("全画面で見る場合は、下の「レポートと設定をダウンロード」でファイルを保存し、ブラウザで開いてください。")

In [ ]:
#@title レポートと設定をダウンロード（任意）
from google.colab import files
files.download("mmm_output/report.html")
files.download(CONFIG)

In [ ]:
#@title ⑤ シナリオ試算（④ の結果を使うので再推定なし・数秒）
#@markdown `チャネル名:倍率` をカンマ区切り（例: `tv:1.5, social:0.8`）。書かないチャネルは現状維持
シナリオ名 = "TV +50%" #@param {type:"string"}
倍率 = "tv:1.5" #@param {type:"string"}
対象週数 = 52 #@param {type:"integer"}

from mmm.colab import parse_multipliers
save_yaml({"scenarios": [{"name": シナリオ名, "period_weeks": 対象週数,
                          "multipliers": parse_multipliers(倍率)}]}, "colab_scenario.yaml")
r = subprocess.run([sys.executable, "-m", "mmm", "simulate", "--config", CONFIG,
                    "--scenario-file", "colab_scenario.yaml"], capture_output=True, text=True)
if r.returncode != 0:
    print(r.stdout[-2000:], r.stderr[-2000:])
    raise SystemExit("❌ エラー")
res = pd.read_csv("mmm_output/scenarios.csv")
tot = res[res["channel"] == "TOTAL"].iloc[0]
print(f"支出 {tot['spend_change']:+,.0f} → {KPIの名前}の増分 {tot['delta_contribution']:+,.0f}{KPIの単位}"
      f"（区間 {tot['delta_lo']:+,.0f} 〜 {tot['delta_hi']:+,.0f}、プラスになる確率 {tot['prob_delta_positive']:.0%}）")
res[["channel", "current_spend", "scenario_spend", "delta_contribution", "delta_lo", "delta_hi",
     "incremental_roi"]].round(1)